[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/natrask/AESCAPE/blob/main/notebooks/05_mcp.ipynb)

# Part 5 — MCP

**MCP is the Model Context Protocol**: an open standard for how an LLM application discovers
and calls tools that live outside its own process.

In Parts 1–4 every tool was a Python function in the same process as the agent loop. That works in a notebook. It does not describe a real simulation stack, where the solver is a compiled binary on a cluster, behind a scheduler, written by someone else. Further, many are starting to look at developing agents that drive a *physical system*; for example, a 3D printer, an autonomous chemistry system, or a microscope. MCP addresses the need for an encapsulation of how agents can discover and run tools - imagine Captain Kirk saying "computer - what scientific tools are available to me" and then making them go.

MCP splits that in two:

- a **server** owns the tools and publishes their names, descriptions and JSON schemas;
- a **client** connects, asks what is available, and calls what it needs.

The idea is that, if one writes their solver's tool surface once as a server, then anything that speaks MCP can drive it.

**What does this notebook do?** We write a small MCP server exposing two tools — a calculator and a scikit-fem Poisson solve — run it as a separate process, and connect to it twice: once by hand, to see the discovery step, and once from a ReAct loop where the model chooses which tool to call.

**You are done when** the model answers a Poisson question by calling `solve_poisson` over the protocol and reporting the degree-of-freedom count it got back.

## 5.0 API Setup

Same key as Part 0, plus the `mcp` and `scikit-fem` packages. `pick_errlog` is a notebook-specific workaround: `stdio_client` needs a stderr stream with a real file descriptor to attach the subprocess to, and the one Jupyter installs does not have one.

In [ ]:
import sys
if 'google.colab' in sys.modules:
    %pip install -U -q mcp scikit-fem "google-genai<2.13" "google-auth==2.49.0"

import os, json, time
from google import genai
from google.genai import types as gtypes
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')

assert API_KEY, "Set GEMINI_API_KEY in Colab secrets or your environment."

gemini = genai.Client(api_key=API_KEY)
MODEL = "gemini-3.1-flash-lite"


def generate_with_retry(*, contents, config=None, max_attempts=6):
    delay = 4.0
    for attempt in range(max_attempts):
        try:
            return gemini.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            msg = str(e)
            if "429" not in msg and "RESOURCE_EXHAUSTED" not in msg and "quota" not in msg.lower():
                raise
            if attempt == max_attempts - 1:
                raise
            print(f"[rate limit] attempt {attempt+1}: sleeping {delay:.1f}s and retrying...")
            time.sleep(delay)
            delay = min(delay * 2, 60.0)



def pick_errlog():
    """stdio_client attaches the server's stderr to a real file descriptor.
    A notebook's sys.stderr has none, so fall back to the process's original."""
    for s in (sys.__stderr__, sys.stderr):
        try:
            if s is not None:
                s.fileno()
                return s
        except Exception:
            continue
    return open(os.devnull, "w")


ERRLOG = pick_errlog()
print("ready.")

## 5.1 The server

An MCP server is a plain Python file. You create an `MCPServer`, decorate each function you want to expose with `@mcp.tool()`, and call `mcp.run()`. The SDK derives every tool's public contract from the function signature: **type hints become the JSON schema**, and the **docstring becomes the description** the model reads. That contract is all a client ever sees.

`mcp.run(transport="stdio")` makes the server speak over stdin and stdout, which is what lets a client launch it as a subprocess. The alternative, `streamable-http`, serves over a socket when the server lives on another machine.

### What we are exposing

Two tools. The first is the `calculator` from Part 1, kept so there is something familiar to compare against.

The second wraps a finite element solve. We use [scikit-fem](https://scikit-fem.readthedocs.io/), a small finite element library written in pure Python — no compiler and no external mesh generator, so it installs with `pip` and runs in Colab. `solve_poisson` solves Poisson's equation

$$-\nabla^2 u = 1 \quad \text{in } \Omega, \qquad u = 0 \quad \text{on } \partial\Omega$$

on an L-shaped domain, which is a standard finite element test problem. The steps in the code are the usual ones: build a triangular mesh, choose piecewise-linear elements, assemble the stiffness matrix and load vector, eliminate the boundary degrees of freedom, and solve the resulting linear system. The `refine` argument sets how many times the mesh is uniformly subdivided, trading accuracy against cost.

A solver like this is the case MCP exists for. It is expensive to run, it holds state — meshes, matrices — that you would not want to serialise, and in practice it often lives on a different machine from whatever is driving it.

Two things to notice in the code:

- `op` is typed `Literal["add", ...]`, not `str`. That produces an `enum` in the published schema, and the server then **rejects** anything else before your function runs. A plain `str` would let bad values through to your code.
- `solve_poisson` returns numbers, not a mesh. Everything crossing the boundary has to be JSON, so the mesh stays server-side and the client gets a summary: the number of degrees of freedom, the number of elements, and the peak value of the solution.

(Tools are one of three MCP primitives. There are also *resources* — read-only data addressed by URI — and *prompts*. This notebook only uses tools.)

The cell below starts with `%%writefile`, a Jupyter magic that saves the cell to disk instead of running it. Nothing executes here — it just creates `aescape_mcp_server.py` next to the notebook, ready for the client to launch as a subprocess.

In [ ]:
%%writefile aescape_mcp_server.py
from typing import Literal

from mcp.server.mcpserver import MCPServer

import numpy as np
from skfem import (MeshTri, Basis, ElementTriP1, BilinearForm, LinearForm,
                   condense, solve)
from skfem.helpers import dot, grad

mcp = MCPServer(name="aescape-tools")


@BilinearForm
def stiffness(u, v, w):
    return dot(grad(u), grad(v))


@LinearForm
def unit_load(v, w):
    return 1.0 * v


@mcp.tool()
def calculator(op: Literal["add", "sub", "mul", "div"], a: float, b: float) -> dict:
    """Perform one arithmetic operation on two numbers."""
    ops = {"add": a + b, "sub": a - b, "mul": a * b,
           "div": a / b if b != 0 else float("nan")}
    return {"result": ops[op]}


@mcp.tool()
def solve_poisson(refine: int = 3) -> dict:
    """Solve -laplace(u) = 1 on the L-shaped domain with u = 0 on the boundary.

    refine: number of uniform refinements, 0 to 6. Higher is finer and slower.
    """
    if not 0 <= refine <= 6:
        return {"error": f"refine must be between 0 and 6; got {refine}"}
    mesh = MeshTri.init_lshaped()
    for _ in range(refine):
        mesh = mesh.refined()
    basis = Basis(mesh, ElementTriP1())
    K = stiffness.assemble(basis)
    f = unit_load.assemble(basis)
    u = solve(*condense(K, f, D=mesh.boundary_nodes()))
    return {"refine": refine,
            "dofs": int(basis.N),
            "elements": int(mesh.nelements),
            "u_max": float(np.max(u))}


if __name__ == "__main__":
    mcp.run(transport="stdio")

## 5.2 Driving it by hand

`stdio` transport launches the server as a subprocess and talks to it over stdin/stdout. (`streamable-http` is the alternative when the server lives on another machine.)

`list_tools()` is the discovery step: the client learns the tool surface at runtime instead of having it hard-coded. The schema it prints is exactly what a model would be shown.

In [ ]:
SERVER = os.path.abspath("aescape_mcp_server.py")


def connect():
    return stdio_client(
        StdioServerParameters(command=sys.executable, args=[SERVER]), errlog=ERRLOG)


async def by_hand():
    async with connect() as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            for t in (await session.list_tools()).tools:
                print(f"{t.name}\n  {(t.description or '').splitlines()[0]}")
                print(f"  schema: {json.dumps(t.input_schema['properties'])}\n")

            r = await session.call_tool("calculator", {"op": "mul", "a": 13, "b": 47})
            print("calculator(mul, 13, 47)   ->", r.content[0].text.replace("\n", " "))

            r = await session.call_tool("solve_poisson", {"refine": 4})
            print("solve_poisson(refine=4)   ->", r.content[0].text.replace("\n", " "))

            # The enum is enforced by the server: this never reaches our function.
            r = await session.call_tool("calculator", {"op": "bogus", "a": 1, "b": 2})
            print("calculator(bogus, ...)    -> is_error =", r.is_error)
            detail = next((l.strip() for l in r.content[0].text.splitlines()
                            if "Input should be" in l), "")
            print("   ", detail)

await by_hand()

## 5.3 Driving it with a model

Calling tools by name with arguments we chose is remote procedure calling; the schema is doing no work. It earns its place when a model reads it and decides for itself.

The conversion is trivial: an MCP `input_schema` is already JSON Schema, and that is exactly what Gemini's `FunctionDeclaration` expects, so it passes through **unmodified**. The server's contract becomes the model's tool list with no translation layer.

The loop below is the ReAct loop from Part 1. The only difference is where the tools came from — discovered over a protocol at runtime, rather than written into this file.

Both cells use `await`; see §3.4 if that is unfamiliar.

In [ ]:
async def ask(question, max_steps=6):
    async with connect() as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # MCP schemas drop straight into the model's tool list.
            decls = [gtypes.FunctionDeclaration(name=t.name,
                                                description=t.description,
                                                parameters=t.input_schema)
                     for t in (await session.list_tools()).tools]
            cfg = gtypes.GenerateContentConfig(
                system_instruction="Use the provided tools. Never compute in your head.",
                tools=[gtypes.Tool(function_declarations=decls)])

            contents = [gtypes.Content(role="user",
                                       parts=[gtypes.Part.from_text(text=question)])]
            for _ in range(max_steps):
                resp = generate_with_retry(contents=contents, config=cfg)
                parts = resp.candidates[0].content.parts or []
                contents.append(resp.candidates[0].content)

                calls = [p.function_call for p in parts if getattr(p, "function_call", None)]
                if not calls:
                    return "".join(getattr(p, "text", "") or "" for p in parts)

                obs = []
                for fc in calls:
                    args = dict(fc.args or {})
                    print(f"  MCP call: {fc.name}({args})")
                    r = await session.call_tool(fc.name, args)
                    text = r.content[0].text
                    try:
                        payload = json.loads(text)
                    except json.JSONDecodeError:
                        payload = {"error": text}
                    obs.append(gtypes.Part.from_function_response(
                        name=fc.name, response=payload))
                contents.append(gtypes.Content(role="user", parts=obs))
            return "(max_steps reached)"


print(await ask("What is 13 * 47 + 8?"))
print()
print(await ask("Solve the Poisson problem with 5 refinements. How many degrees of freedom?"))

## 5.4 Syntax summary

**Server**

| | |
|---|---|
| `from mcp.server.mcpserver import MCPServer` | the server class |
| `mcp = MCPServer(name="...")` | create it |
| `@mcp.tool()` | publish a function; hints → schema, docstring → description |
| `Literal[...]` on a parameter | becomes an `enum`; the server enforces it |
| `mcp.run(transport="stdio")` | serve over stdin/stdout (or `"streamable-http"`) |

**Client**

| | |
|---|---|
| `StdioServerParameters(command=..., args=[...])` | how to launch the server |
| `async with stdio_client(params, errlog=...)` | open the transport |
| `async with ClientSession(read, write) as session` | open a session |
| `await session.initialize()` | handshake |
| `await session.list_tools()` | discover — `.name`, `.description`, `.input_schema` |
| `await session.call_tool(name, {...})` | invoke — `.content[0].text`, `.is_error` |

Three things worth knowing:

- **The SDK renamed things in 2.x.** `FastMCP` is now `MCPServer`, and `inputSchema` is now
  `input_schema`. Most tutorials online are still 1.x. Pin `mcp<2` for the old API.
- **Never `print()` inside a stdio tool.** stdout *is* the protocol channel; a stray print
  corrupts the stream. Log to stderr instead.
- **In a notebook, pass `errlog=` explicitly.** It defaults to `sys.stderr`, which the kernel
  replaces with an object having no file descriptor — you get `UnsupportedOperation: fileno`.

## 5.5 Further reading

- [modelcontextprotocol.io](https://modelcontextprotocol.io) — specification and concepts
- [Python SDK](https://github.com/modelcontextprotocol/python-sdk) — source and examples
- [2.x migration guide](https://py.sdk.modelcontextprotocol.io/v2/migration/) — what changed from 1.x
- [Reference servers](https://github.com/modelcontextprotocol/servers) — filesystem, git, databases and more